# Live Demo

In [ ]:
import sys
import os
import open3d as o3d
import copy
import numpy as np

sys.path.append(os.path.abspath("../src"))
from camera import Camera
from config import RegistrationConfig, load_config_yaml
from dataset_loader import DatasetLoader
from evaluation import Evaluation, compare_evaluations, get_better_eval
from mask import masked_image
from paths import PathManager
from pose_initializer import PoseInitializer
from segmentation import predict_segment
from template import TemplateLoader
from utils import show_image, scene_from_depth

In [ ]:
item = "apple"
idx = 300
config = "../configs/baseline.yaml"
cam_name = "D435i"

cfg = load_config_yaml(config)

pm = PathManager(item)
cam = Camera.from_yaml(pm.get_camera_path(), cam_name)
loader = TemplateLoader(pm)
templates = loader.load_all()

dl = DatasetLoader(item)

In [ ]:
rgb, depth, t_gt = dl(idx)
try:
    pts, seg_img = predict_segment(rgb, item)
except TargetNotFoundError as e:
    print(f"[WARNING] [idx {idx}] {e}")
except SegmentationError as e:
    print(f"[ERROR] [idx {idx}] Segmentation failed: {e}")
masked_depth = masked_image(pts, depth)
scene = scene_from_depth(cam, masked_depth)
full_scene = scene_from_depth(cam, depth)

In [ ]:
show_image(rgb, "RGB-Image", convert_to_bgr=True)

In [ ]:
show_image(seg_img, "Segmentation-Image", convert_to_bgr=True)

In [ ]:
show_image(depth, "Depth-Image")

In [ ]:
show_image(masked_depth, "Masked Depth-Image")

In [ ]:
pose_init = PoseInitializer(cfg, item, cam)
best_ev = None
t_full = None
for template in templates:
    transformation, ev = pose_init.process_one(scene, template.pcd, t_gt)

    # o3d.visualization.draw_geometries([template.pcd], window_name=template.name)
    chosen = get_better_eval(ev, best_ev)
    if chosen is ev:
        best_template = template
        best_transformation = transformation
        best_ev = ev

    if template.name == "full_model":
        t_full = transformation

In [ ]:
RED = [1, 0, 0]
GREEN = [0, 1, 0]
BLUE = [0, 0.2, 1]
# GRAY = [0.3, 0.3, 0.3]
GRAY = [0.5, 0.5, 0.5]
BLACK = [0, 0, 0]

In [ ]:
scene.paint_uniform_color(GREEN)
full_scene.paint_uniform_color(BLACK)
model = copy.deepcopy(next(t for t in templates if t.name == "full_model").pcd)
model.transform(t_gt)
model.paint_uniform_color(GRAY)

In [ ]:
if t_full is not None:
    full_template = copy.deepcopy(next(t for t in templates if t.name == "full_model").pcd)
    full_template.transform(t_full)
    full_template.paint_uniform_color(RED)

if best_transformation is not None:
    partial_template = copy.deepcopy(next(t for t in templates if t.name == best_template.name).pcd)
    partial_template.transform(best_transformation)
    partial_template.paint_uniform_color(BLUE)

In [ ]:
print(f"BEST TRANSFORMATION\n{best_transformation}")

In [ ]:
# Evaluation
print(f"BEST TEMPLATE {ev}")

In [ ]:
# ALL: Full-Scene + Scene + Model + 
o3d.visualization.draw_geometries([full_scene, scene, model, full_template, partial_template], window_name=f"idx:{idx}, full_model and {best_template.name}")

In [ ]:
# Scene + Template
# o3d.visualization.draw_geometries([scene, partial_template], window_name=f"idx:{idx}, full_model and {best_template.name}")

# Full-Scene + Scene + Template
# o3d.visualization.draw_geometries([full_scene, scene, partial_template], window_name=f"idx:{idx}, full_model and {best_template.name}")

# Scene + Full-Template
# o3d.visualization.draw_geometries([scene, full_template], window_name=f"idx:{idx}, full_model and {best_template.name}")

# Full-Scene + Scene + Full-Template
# o3d.visualization.draw_geometries([full_scene, scene, full_template], window_name=f"idx:{idx}, full_model and {best_template.name}")